# Mundial 2026 – Modelo Predictivo (Robberechts et al.)

> Versión mejorada: datos desde 1990, validación leave-one-out en 2018/2022, y parámetros optimizados propagados a todo el pipeline.

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from collections import Counter


## 2. Cargar y limpiar datos

**MEJORA 1 – Datos desde 1990 en lugar de 2010.**  
Empezar en 2010 hace que el Elo inicial de todas las selecciones sea 1500, ignorando décadas de historial. Con datos desde 1990 los ratings de equipos como Brasil, Argentina o España llegan al 2026 con un punto de partida más realista.  

El dataset de Kaggle (`results.csv`) incluye partidos desde 1872; usaremos 1990 como compromiso entre calidad de datos y coste computacional.

In [2]:
# from google.colab import drive
# drive.mount('/content/drive')

In [4]:
df = pd.read_csv(
    "../data/results.csv",
    parse_dates=["date"]
)

# df = pd.read_csv(
#     "/content/drive/MyDrive/fottball_data_wordlcup/results.csv",
#     parse_dates=["date"]
# )

# Eliminar partidos de un equipo consigo mismo (datos corruptos)
df = df[df["home_team"] != df["away_team"]]
df = df.sort_values("date").reset_index(drop=True)

historical_df = df.dropna(subset=["home_score", "away_score"]).copy()
future_matches = df[df["home_score"].isna()].copy()

historical_df["home_score"] = historical_df["home_score"].astype(int)
historical_df["away_score"] = historical_df["away_score"].astype(int)

# ── MEJORA 1: desde 1990, no desde 2010 ──────────────────────────────────────
historical_df = historical_df[historical_df["date"] >= "1990-01-01"]

print(f"Primer partido : {historical_df['date'].min().date()}")
print(f"Último partido : {historical_df['date'].max().date()}")
print(f"Partidos históricos: {len(historical_df):,}")
print(f"Partidos futuros   : {len(future_matches):,}")


Primer partido : 1990-01-12
Último partido : 2026-05-31
Partidos históricos: 32,164
Partidos futuros   : 72


## 3. Pesos de competición

In [5]:
INITIAL_ELO  = 1500
INITIAL_ODM  = 1.0

TOURNAMENT_W = {
    "Friendly":                              0.33,
    "FIFA World Cup qualification":          0.45,
    "UEFA Euro qualification":               0.50,
    "African Cup of Nations qualification":  0.45,
    "AFC Asian Cup qualification":           0.45,
    "CONCACAF Nations League":               0.60,
    "Gold Cup":                              0.60,
    "AFC Asian Cup":                         0.65,
    "African Cup of Nations":                0.70,
    "UEFA Nations League":                   0.80,
    "Copa América":                          0.80,
    "UEFA Euro":                             0.85,
    "FIFA World Cup":                        1.00,
}

def get_w_factor(tournament):
    return TOURNAMENT_W.get(tournament, 0.50)


## 4. Sistema Elo

In [6]:
def expected_score(rating_a, rating_b):
    return 1.0 / (1.0 + 10.0 ** ((rating_b - rating_a) / 400.0))

def actual_score(home_goals, away_goals):
    if home_goals > away_goals: return 1.0, 0.0
    if home_goals < away_goals: return 0.0, 1.0
    return 0.5, 0.5

def update_elo(home_r, away_r, home_g, away_g, tournament, k0, gamma):
    exp_h = expected_score(home_r, away_r)
    act_h, act_a = actual_score(home_g, away_g)
    delta = abs(home_g - away_g)
    k = k0 * get_w_factor(tournament) * (1 + delta) ** gamma
    return (home_r + k * (act_h - exp_h),
            away_r + k * (act_a - (1 - exp_h)))


## 5. Modelo ODM (Offense-Defense)

In [7]:
def scale(A, x):
    x_inv  = 1.0 / np.maximum(x, 1e-6)
    y      = A @ x_inv
    y_inv  = 1.0 / np.maximum(y, 1e-6)
    x_post = A.T @ y_inv
    y_post = A @ (1.0 / np.maximum(x_post, 1e-6))
    return x_post, y_post

def compute_post_game_odm(o_pre, d_pre, home_g, away_g, nb_iter=10, eps=0.01):
    A = np.array([
        [eps,          away_g + eps],
        [home_g + eps, eps         ]
    ])
    o_post = np.array(o_pre, dtype=float)
    d_post = np.array(d_pre, dtype=float)
    for _ in range(nb_iter):
        o_r_o, d_r_o = scale(A,   o_post)
        d_r_d, o_r_d = scale(A.T, d_post)
        o_post = o_r_o + o_r_d
        d_post = d_r_o + d_r_d
    return o_post[0], o_post[1], d_post[0], d_post[1]


## 6. Cálculo del historial combinado Elo + ODM

In [8]:
def calculate_combined_history(matches, k0=40.0, gamma=0.5):
    """
    Recorre los partidos en orden cronológico y devuelve un DataFrame
    con los ratings PRE-partido de cada equipo, más una snapshot final.
    Los índices del DataFrame resultante coinciden con los de 'matches'.
    """
    ratings_elo   = {}
    odm_o_wsum    = {}
    odm_d_wsum    = {}
    odm_w_sum     = {}
    records       = []

    for row in matches.sort_values("date").itertuples():
        home, away   = row.home_team, row.away_team
        h_elo = ratings_elo.get(home, INITIAL_ELO)
        a_elo = ratings_elo.get(away, INITIAL_ELO)

        def _odm(team):
            if odm_w_sum.get(team, 0) > 0:
                return (odm_o_wsum[team] / odm_w_sum[team],
                        odm_d_wsum[team] / odm_w_sum[team])
            return INITIAL_ODM, INITIAL_ODM

        h_o, h_d = _odm(home)
        a_o, a_d = _odm(away)

        records.append({
            "date":       row.date,
            "home_team":  home,      "away_team":  away,
            "home_score": row.home_score, "away_score": row.away_score,
            "home_elo":   h_elo,     "away_elo":   a_elo,
            "home_o":     h_o,       "home_d":     h_d,
            "away_o":     a_o,       "away_d":     a_d,
            "tournament": row.tournament,
            "neutral":    row.neutral,
        })

        # Actualizar Elo
        new_h_elo, new_a_elo = update_elo(
            h_elo, a_elo, row.home_score, row.away_score,
            row.tournament, k0, gamma)
        ratings_elo[home] = new_h_elo
        ratings_elo[away] = new_a_elo

        # Actualizar ODM
        h_o_p, a_o_p, h_d_p, a_d_p = compute_post_game_odm(
            [h_o, a_o], [h_d, a_d], row.home_score, row.away_score)
        wi = get_w_factor(row.tournament)
        for team, o_p, d_p in [(home, h_o_p, h_d_p), (away, a_o_p, a_d_p)]:
            odm_o_wsum[team] = odm_o_wsum.get(team, 0.0) + o_p * wi
            odm_d_wsum[team] = odm_d_wsum.get(team, 0.0) + d_p * wi
            odm_w_sum[team]  = odm_w_sum.get(team,  0.0) + wi

    df_out = pd.DataFrame(records, index=matches.sort_values("date").index)
    return df_out


## 7. Métricas de evaluación (RPS y Accuracy)

In [9]:
def calculate_rps(probs, actual_results):
    """
    probs: (N,3) – [p_derrota, p_empate, p_victoria]
    actual_results: (N,)  – 0=derrota, 1=empate, 2=victoria
    """
    N = len(actual_results)
    y = np.zeros((N, 3))
    y[np.arange(N), actual_results] = 1.0
    cum_p = np.cumsum(probs,     axis=1)
    cum_y = np.cumsum(y,         axis=1)
    return np.mean(np.sum((cum_p[:, :2] - cum_y[:, :2]) ** 2, axis=1) / 2.0)

def calculate_accuracy(probs, actual_results):
    predicted = np.argmax(probs, axis=1)
    return np.mean(predicted == actual_results)

def encode_result(home_score, away_score):
    if home_score < away_score: return 0
    if home_score == away_score: return 1
    return 2


## 8. Regresión Logit Ordenada

In [10]:
def ordered_logit_probs(X_dot_beta, theta_0, d_theta):
    """
    theta_0  : primer punto de corte (derrota | empate)
    d_theta  : log de la distancia al segundo corte (garantiza theta_1 > theta_0)
    """
    theta_1   = theta_0 + np.exp(d_theta)
    p_loss    = 1.0 / (1.0 + np.exp(-(theta_0 - X_dot_beta)))
    p_loss_tie= 1.0 / (1.0 + np.exp(-(theta_1 - X_dot_beta)))
    p_tie     = p_loss_tie - p_loss
    p_win     = 1.0 - p_loss_tie
    probs     = np.column_stack((p_loss, p_tie, p_win))
    return np.clip(probs, 1e-15, 1.0)

def features_from_df(df):
    elo_diff = (df["home_elo"] - df["away_elo"]) / 100.0
    odm_o_d  = df["home_o"] - df["away_d"]
    odm_d_o  = df["home_d"] - df["away_o"]
    home_adv = (~df["neutral"]).astype(float)
    return elo_diff.values, odm_o_d.values, odm_d_o.values, home_adv.values

def compute_xbeta(df, beta_elo, beta_odm1, beta_odm2, beta_home):
    ed, od1, od2, ha = features_from_df(df)
    return beta_elo * ed + beta_odm1 * od1 + beta_odm2 * od2 + beta_home * ha


## 9. Optimización conjunta L-BFGS-B

**MEJORA 2 – Validación leave-one-out en mundiales anteriores.**  
El paper valida en los mundiales 2002-2014 con leave-one-out. Aquí usamos los mundiales de **2018 y 2022** como conjunto de validación para calibrar los parámetros antes de predecir 2026. Esto garantiza que `k0`, `gamma` y las betas del logit no estén sobreajustadas a un solo torneo.

**MEJORA 3 – Parámetros optimizados propagados a todo el pipeline.**  
La función objetivo recalcula el historial completo en cada iteración del optimizador (tal como indica el paper), de modo que `k0` y `gamma` se optimizan conjuntamente con las betas de la regresión.


In [11]:
def objective_function(params, historical_matches, validation_indices):
    k0, gamma, b_elo, b_odm1, b_odm2, b_home, theta_0, d_theta = params

    # Recalcular historial completo con los nuevos k0/gamma
    df_hist = calculate_combined_history(historical_matches, k0=k0, gamma=gamma)

    # Extraer solo el set de validación
    val_df  = df_hist.loc[validation_indices].copy()

    X_beta  = compute_xbeta(val_df, b_elo, b_odm1, b_odm2, b_home)
    probs   = ordered_logit_probs(X_beta, theta_0, d_theta)
    y_true  = np.array([encode_result(r.home_score, r.away_score)
                        for r in val_df.itertuples()])
    return calculate_rps(probs, y_true)


def train_joint_model(historical_matches, validation_indices):
    """
    Optimiza conjuntamente k0, gamma y los 6 parámetros del ordered logit.
    Devuelve un dict con los parámetros óptimos.
    """
    # [k0, gamma, beta_elo, beta_odm1, beta_odm2, beta_home, theta_0, d_theta]
    x0 = [40.0, 0.5, 0.5, 0.0, 0.0, 0.3, -0.5, 0.0]
    bounds = [
        (5.0,  120.0),   # k0
        (0.0,    2.0),   # gamma
        (None, None),    # beta_elo
        (None, None),    # beta_odm1
        (None, None),    # beta_odm2
        (0.0,  None),    # beta_home  (la ventaja local debe ser >= 0)
        (None, None),    # theta_0
        (-5.0,   5.0),   # d_theta
    ]

    print("Iniciando optimización L-BFGS-B… puede tardar unos minutos.")
    res = minimize(
        objective_function,
        x0=x0,
        args=(historical_matches, validation_indices),
        method="L-BFGS-B",
        bounds=bounds,
        options={"disp": True, "maxiter": 100},
    )
    print(f"Optimización completada. RPS de validación: {res.fun:.4f}")

    keys = ["k0", "gamma", "beta_elo", "beta_odm1",
            "beta_odm2", "beta_home", "theta_0", "d_theta"]
    return dict(zip(keys, res.x))


## 10. Definir índices de validación y entrenar

**MEJORA 2 (aplicada):** usamos los mundiales 2018 **y** 2022 como validación.  
Si quieres replicar exactamente el leave-one-out del paper, puedes añadir 2014 también.


In [12]:
# Mundiales 2018 y 2022 como set de validación
wc2018_mask = (
    (historical_df["date"] >= "2018-06-14") &
    (historical_df["date"] <= "2018-07-15") &
    (historical_df["tournament"] == "FIFA World Cup")
)
wc2022_mask = (
    (historical_df["date"] >= "2022-11-20") &
    (historical_df["date"] <= "2022-12-18") &
    (historical_df["tournament"] == "FIFA World Cup")
)

validation_indices = historical_df[wc2018_mask | wc2022_mask].index
print(f"Partidos de validación (WC 2018 + 2022): {len(validation_indices)}")

# Entrenar
optimal_params = train_joint_model(historical_df, validation_indices)
print("\nParámetros óptimos:")
for k, v in optimal_params.items():
    print(f"  {k:<12} = {v:.6f}")


Partidos de validación (WC 2018 + 2022): 128
Iniciando optimización L-BFGS-B… puede tardar unos minutos.


C:\Users\daniel.cejas\AppData\Local\Temp\ipykernel_25596\2527869399.py:36: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  res = minimize(


Optimización completada. RPS de validación: 0.1942

Parámetros óptimos:
  k0           = 32.474769
  gamma        = 0.000000
  beta_elo     = 0.780465
  beta_odm1    = 0.032013
  beta_odm2    = -0.030305
  beta_home    = 0.653803
  theta_0      = 1.121959
  d_theta      = 0.160535


## 11. Calcular ratings finales con parámetros óptimos

In [13]:
# ── MEJORA 3: usar k0 y gamma del optimizador, no los defaults ──────────────
print("Calculando historial final con parámetros óptimos…")
df_history_final = calculate_combined_history(
    historical_df,
    k0    = optimal_params["k0"],
    gamma = optimal_params["gamma"],
)

# Snapshot: último rating conocido de cada selección
latest_ratings = {}
all_teams = pd.concat([
    df_history_final["home_team"],
    df_history_final["away_team"]
]).unique()

for team in all_teams:
    mask = (
        (df_history_final["home_team"] == team) |
        (df_history_final["away_team"] == team)
    )
    last = df_history_final[mask].iloc[-1]
    if last["home_team"] == team:
        latest_ratings[team] = {
            "elo": last["home_elo"],
            "o":   last["home_o"],
            "d":   last["home_d"],
        }
    else:
        latest_ratings[team] = {
            "elo": last["away_elo"],
            "o":   last["away_o"],
            "d":   last["away_d"],
        }

print(f"Ratings calculados para {len(latest_ratings)} selecciones.")

# Top 20
ranking_df = (
    pd.DataFrame([
        {"team": t, "elo": v["elo"], "attack": v["o"], "defense": v["d"]}
        for t, v in latest_ratings.items()
    ])
    .sort_values("elo", ascending=False)
    .reset_index(drop=True)
)
print("\nTOP 20 ELO:")
print(ranking_df.head(20).to_string(index=False))


Calculando historial final con parámetros óptimos…
Ratings calculados para 326 selecciones.

TOP 20 ELO:
       team         elo    attack   defense
      Spain 1948.384348 63.611390  5.365258
  Argentina 1927.897541 42.914214  9.351697
     France 1894.950452 63.765978  5.646681
     Brazil 1853.598519 62.197390  7.378325
    England 1846.352007 55.904599  5.652602
   Portugal 1839.795401 54.065684  6.480917
Netherlands 1829.385560 49.119220  6.010842
    Germany 1824.305642 58.188001  7.006277
   Colombia 1821.845536 29.680398 12.489060
      Japan 1814.779154 30.847698 13.691864
    Morocco 1813.161978 40.595588  9.128056
      Italy 1806.196894 60.087366  4.475578
    Croatia 1789.006603 47.477280  8.608498
     Mexico 1773.779563 49.245173 10.689578
    Uruguay 1773.687382 35.042678 11.179917
       Iran 1771.611271 35.528026  9.250463
    Senegal 1769.381708 33.035624  9.490879
    Belgium 1757.232458 35.368110 11.221626
South Korea 1756.633587 34.351276 10.865000
  Australia 175

## 12. Funciones de predicción y simulación

In [14]:
def predict_match(team_home, team_away, ratings, params, neutral=True):
    """Devuelve dict con home_win / draw / away_win."""
    h = ratings[team_home]
    a = ratings[team_away]
    elo_diff = (h["elo"] - a["elo"]) / 100.0
    odm_o_d  = h["o"]   - a["d"]
    odm_d_o  = h["d"]   - a["o"]
    home_adv = 0.0 if neutral else 1.0

    X = (params["beta_elo"]  * elo_diff +
         params["beta_odm1"] * odm_o_d  +
         params["beta_odm2"] * odm_d_o  +
         params["beta_home"] * home_adv)

    probs = ordered_logit_probs(np.array([X]),
                                params["theta_0"],
                                params["d_theta"])[0]
    return {"away_win": probs[0], "draw": probs[1], "home_win": probs[2]}


def simulate_group_match(home, away, ratings, params):
    """Simula un partido de grupos (puede terminar en empate)."""
    p = predict_match(home, away, ratings, params, neutral=True)
    return np.random.choice(
        ["away_win", "draw", "home_win"],
        p=[p["away_win"], p["draw"], p["home_win"]]
    )


def simulate_knockout_match(team_a, team_b, ratings, params):
    """Simula un partido eliminatorio (50/50 en los penaltis)."""
    p = predict_match(team_a, team_b, ratings, params, neutral=True)
    pa = p["home_win"] + 0.5 * p["draw"]
    pb = p["away_win"] + 0.5 * p["draw"]
    total = pa + pb
    return np.random.choice([team_a, team_b], p=[pa / total, pb / total])


## 13. Generador de marcadores para la fase de grupos

**MEJORA 4 – Distribución de goles basada en Poisson.**  
El `sample_score` original usaba distribuciones manuales con probabilidades fijas. Un enfoque más riguroso es usar distribuciones de Poisson cuya media depende de los ratings ODM de cada equipo. Esto hace que el número de goles sea consistente con las fortalezas ofensivas/defensivas reales, lo que mejora el desempate por diferencia de goles.


In [15]:
def sample_score_poisson(home, away, ratings, result):
    """
    Genera un marcador coherente con el resultado sorteado (result).
    Las medias de gol se escalan a partir de los ratings ODM.
    """
    h = ratings[home]
    a = ratings[away]

    # Media base ~1.3 goles por equipo, modulada por ataque vs defensa
    BASE = 1.3
    lambda_h = max(BASE * h["o"] / max(a["d"], 0.1), 0.3)
    lambda_a = max(BASE * a["o"] / max(h["d"], 0.1), 0.3)

    # Samplear hasta obtener un marcador consistente con el resultado
    for _ in range(200):
        gh = np.random.poisson(lambda_h)
        ga = np.random.poisson(lambda_a)
        if result == "home_win" and gh > ga:  return gh, ga
        if result == "away_win" and ga > gh:  return gh, ga
        if result == "draw"     and gh == ga: return gh, ga

    # Fallback determinista si el muestreo no converge
    if result == "home_win": return 1, 0
    if result == "away_win": return 0, 1
    return 0, 0


## 14. Simulación fase de grupos

In [16]:
def create_group_table(teams):
    return {t: {"pts": 0, "gf": 0, "ga": 0, "gd": 0} for t in teams}

def update_table(table, home, away, hg, ag):
    table[home]["gf"] += hg;  table[home]["ga"] += ag
    table[away]["gf"] += ag;  table[away]["ga"] += hg
    table[home]["gd"]  = table[home]["gf"] - table[home]["ga"]
    table[away]["gd"]  = table[away]["gf"] - table[away]["ga"]
    if   hg > ag: table[home]["pts"] += 3
    elif ag > hg: table[away]["pts"] += 3
    else:         table[home]["pts"] += 1; table[away]["pts"] += 1

def simulate_group(teams, ratings, params):
    table = create_group_table(teams)
    for i in range(len(teams)):
        for j in range(i + 1, len(teams)):
            home, away  = teams[i], teams[j]
            result      = simulate_group_match(home, away, ratings, params)
            # ── MEJORA 4 aplicada ─────────────────────────────────────────
            hg, ag      = sample_score_poisson(home, away, ratings, result)
            update_table(table, home, away, hg, ag)

    # Desempate: puntos → DG → GF → Elo (último recurso)
    ranking = sorted(
        table.items(),
        key=lambda x: (x[1]["pts"], x[1]["gd"], x[1]["gf"],
                       ratings[x[0]]["elo"]),
        reverse=True,
    )
    return ranking

def simulate_group_stage(groups, ratings, params):
    winners, runners, thirds = {}, {}, []
    for name, teams in groups.items():
        rank = simulate_group(teams, ratings, params)
        winners[name]  = rank[0][0]
        runners[name]  = rank[1][0]
        thirds.append({
            "group":  name,
            "team":   rank[2][0],
            "pts":    rank[2][1]["pts"],
            "gd":     rank[2][1]["gd"],
            "gf":     rank[2][1]["gf"],
        })
    return {"winners": winners, "runners_up": runners, "thirds": thirds}

def select_best_thirds(thirds, ratings):
    return sorted(
        thirds,
        key=lambda x: (x["pts"], x["gd"], x["gf"], ratings[x["team"]]["elo"]),
        reverse=True,
    )[:8]


## 15. Cuadro eliminatorio (Mundial 48 equipos)

In [17]:
def get_third_place_mapping(best_thirds, mapping_df):
    qualified_str = ",".join(sorted(x["group"] for x in best_thirds))
    row = mapping_df[mapping_df["Grupos_Clasificados"] == qualified_str]
    if len(row) == 0:
        raise ValueError(f"Sin mapeo para: {qualified_str}")
    return row.iloc[0]

def build_round_of_32(group_results, best_thirds, mapping_row):
    W = group_results["winners"]
    R = group_results["runners_up"]
    T = {x["group"]: x["team"] for x in best_thirds}

    def t(col):
        return T[mapping_row[col][1]]

    return {
        73:  (R["A"],      R["B"]),
        74:  (W["E"],      t("Rival_de_1E")),
        75:  (W["F"],      R["C"]),
        76:  (W["C"],      R["F"]),
        77:  (W["I"],      t("Rival_de_1I")),
        78:  (R["E"],      R["I"]),
        79:  (W["A"],      t("Rival_de_1A")),
        80:  (W["L"],      t("Rival_de_1L")),
        81:  (W["D"],      t("Rival_de_1D")),
        82:  (W["G"],      t("Rival_de_1G")),
        83:  (R["K"],      R["L"]),
        84:  (W["H"],      R["J"]),
        85:  (W["B"],      t("Rival_de_1B")),
        86:  (W["J"],      R["H"]),
        87:  (W["K"],      t("Rival_de_1K")),
        88:  (R["D"],      R["G"]),
    }

def build_round_of_16(w32):
    return {
        89: (w32[74], w32[77]),  90: (w32[73], w32[75]),
        91: (w32[76], w32[78]),  92: (w32[79], w32[80]),
        93: (w32[83], w32[84]),  94: (w32[81], w32[82]),
        95: (w32[86], w32[88]),  96: (w32[85], w32[87]),
    }

def build_quarters(w16):
    return {
        97:  (w16[89], w16[90]),  98:  (w16[93], w16[94]),
        99:  (w16[91], w16[92]),  100: (w16[95], w16[96]),
    }

def build_semis(w8):
    return {101: (w8[97], w8[98]), 102: (w8[99], w8[100])}

def build_final(w4):
    return {104: (w4[101], w4[102])}

def play_match_dict(matches, ratings, params):
    return {
        mid: simulate_knockout_match(ta, tb, ratings, params)
        for mid, (ta, tb) in matches.items()
    }


## 16. Simulación completa del Mundial

In [18]:
def simulate_world_cup(groups, ratings, params, mapping_df):
    gs       = simulate_group_stage(groups, ratings, params)
    thirds   = select_best_thirds(gs["thirds"], ratings)
    mapping  = get_third_place_mapping(thirds, mapping_df)

    w32 = play_match_dict(build_round_of_32(gs, thirds, mapping), ratings, params)
    w16 = play_match_dict(build_round_of_16(w32),                 ratings, params)
    w8  = play_match_dict(build_quarters(w16),                    ratings, params)
    w4  = play_match_dict(build_semis(w8),                        ratings, params)
    return play_match_dict(build_final(w4), ratings, params)[104]


## 17. Grupos del Mundial 2026

In [19]:
groups = {
    "A": ["Mexico",        "South Africa",          "South Korea",  "Czech Republic"],
    "B": ["Canada",        "Bosnia and Herzegovina","Qatar",         "Switzerland"],
    "C": ["Brazil",        "Morocco",               "Haiti",         "Scotland"],
    "D": ["United States", "Paraguay",              "Australia",     "Turkey"],
    "E": ["Germany",       "Curaçao",               "Ivory Coast",   "Ecuador"],
    "F": ["Netherlands",   "Japan",                 "Sweden",        "Tunisia"],
    "G": ["Belgium",       "Egypt",                 "Iran",          "New Zealand"],
    "H": ["Spain",         "Cape Verde",            "Saudi Arabia",  "Uruguay"],
    "I": ["France",        "Senegal",               "Iraq",          "Norway"],
    "J": ["Argentina",     "Algeria",               "Austria",       "Jordan"],
    "K": ["Portugal",      "DR Congo",              "Uzbekistan",    "Colombia"],
    "L": ["England",       "Croatia",               "Ghana",         "Panama"],
}

third_mapping_df = pd.read_csv("../data/combinaciones_terceros_mundial_2026.csv")
# third_mapping_df = pd.read_csv("/content/drive/MyDrive/data/combinaciones_terceros_mundial_2026.csv")



## 18. Simulación Monte Carlo – 10 000 iteraciones

In [20]:
N = 10_000
counter = Counter()

print(f"Iniciando {N:,} simulaciones del Mundial 2026…")
for i in range(N):
    champion = simulate_world_cup(groups, latest_ratings, optimal_params, third_mapping_df)
    counter[champion] += 1
    if (i + 1) % 2000 == 0:
        print(f"  {i+1:,} / {N:,}")

results_df = (
    pd.DataFrame(
        [(team, wins / N) for team, wins in counter.items()],
        columns=["team", "probability"]
    )
    .sort_values("probability", ascending=False)
    .reset_index(drop=True)
)

print("\n=== PROBABILIDADES DE GANAR EL MUNDIAL 2026 ===")
print(results_df.head(20).assign(
    probability=lambda d: d["probability"].map(lambda x: f"{100*x:.2f}%")
).to_string(index=False))


Iniciando 10,000 simulaciones del Mundial 2026…
  2,000 / 10,000
  4,000 / 10,000
  6,000 / 10,000
  8,000 / 10,000
  10,000 / 10,000

=== PROBABILIDADES DE GANAR EL MUNDIAL 2026 ===
       team probability
     France      21.40%
      Spain      15.74%
    Germany      13.83%
  Argentina      10.30%
     Brazil       8.23%
Netherlands       4.81%
   Portugal       3.28%
   Colombia       3.00%
    England       2.75%
    Morocco       2.49%
      Japan       1.96%
     Mexico       1.68%
       Iran       1.21%
    Croatia       1.10%
    Uruguay       1.00%
     Panama       0.98%
South Korea       0.83%
    Senegal       0.75%
  Australia       0.66%
Switzerland       0.63%


## 19. Evaluación retrospectiva en los mundiales de validación

**MEJORA 2 (resultado):** comprobamos que el modelo tiene capacidad predictiva real antes de fiarnos de las simulaciones.


In [21]:
for wc_year, start, end in [
    (2018, "2018-06-14", "2018-07-15"),
    (2022, "2022-11-20", "2022-12-18"),
]:
    mask = (
        (df_history_final["date"] >= start) &
        (df_history_final["date"] <= end)
    )
    val = df_history_final[mask].copy()
    if len(val) == 0:
        print(f"WC{wc_year}: sin datos en el historial final")
        continue

    X_beta = compute_xbeta(val,
                           optimal_params["beta_elo"],
                           optimal_params["beta_odm1"],
                           optimal_params["beta_odm2"],
                           optimal_params["beta_home"])
    probs  = ordered_logit_probs(X_beta,
                                 optimal_params["theta_0"],
                                 optimal_params["d_theta"])
    y_true = np.array([encode_result(r.home_score, r.away_score)
                       for r in val.itertuples()])

    rps = calculate_rps(probs, y_true)
    acc = calculate_accuracy(probs, y_true)
    print(f"WC{wc_year}  |  Partidos: {len(val):>2}  |  RPS: {rps:.4f}  |  Accuracy: {acc:.4f}")


WC2018  |  Partidos: 65  |  RPS: 0.2135  |  Accuracy: 0.5692
WC2022  |  Partidos: 87  |  RPS: 0.2426  |  Accuracy: 0.4943
